In [0]:
dbutils.widgets.text("file_name", "")
dbutils.widgets.text("sas_token", "")
dbutils.widgets.text("environment", "dev")
dbutils.widgets.text("storage_account", "")

storage_account = dbutils.widgets.get("storage_account")
file_name = dbutils.widgets.get("file_name")
sas_token = dbutils.widgets.get("sas_token")
environment = dbutils.widgets.get("environment").lower()


In [0]:
allowed_environments = ["dev", "test", "prod"]

if environment not in allowed_environments:
    raise ValueError(
        f"Invalid environment: {environment}."
        f"Valid values: {allowed_environments}."
    )

In [0]:
container = "data"

url = (
    f"https://{storage_account}.blob.core.windows.net/"
    f"{container}/bronze/weather/{file_name}?{sas_token}"
)

In [0]:
catalog_name = f"weather_{environment}"
schema_name = "silver"
table_name = "weather"

full_table_name = f"{catalog_name}.{schema_name}.{table_name}"

print(f"Full table name: {full_table_name}")

Full table name: weather_dev.silver.weather


In [0]:
import requests

response = requests.get(url, timeout=30)
response.raise_for_status()


payload = response.json()

In [0]:
hours =  payload["forecast"]["forecastday"][0]["hour"]

rows = [
    {
        "timestamp": h["time"],
        "temperature_c": h["temp_c"],
        "humidity": h["humidity"],
        "wind_speed": h["wind_kph"],
        "condition": h["condition"][ "text"]
    }

    for h in hours
]

df = spark.createDataFrame(rows)

In [0]:
from pyspark.sql import functions as F

silver_df = (
    df
    .withColumn("timestamp", F.to_timestamp("timestamp"))
    .withColumn("ingestion_ts", F.current_timestamp())
)

In [0]:
silver_df.write.format("delta").mode("append").saveAsTable(full_table_name)